In [2]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [3]:
master_df = pd.read_csv('master_df_with_deltas.csv')
bestQuestions = ['Q5J', 'Q5E', 'Q3A', 'Q3F', 'Q4E', 'Q3B', 'Q2H', 'Q5B', 'Q3C', 'Q5D']

#bestQuestions = master_df["Question"].isin(bestQuestions)

In [4]:
master_df.columns

Index(['ID', 'Question', 'QuestionText', 'Topic', 'ValidResponse',
       'BeforeNumber', 'BeforeResponse', 'AfterNumber', 'AfterResponse',
       'GROUP', 'RACE', 'AGEBRACKET', 'EDUCATION', 'PARTYBEFORE', 'GENDER',
       'AGE', 'run_0', 'run_1', 'run_2', 'run_3', 'run_4', 'run_5', 'run_6',
       'run_7', 'run_8', 'run_9', 'delta_observed', 'delta_0', 'delta_1',
       'delta_2', 'delta_3', 'delta_4', 'delta_5', 'delta_6', 'delta_7',
       'delta_8', 'delta_9', 'delta_mean_llm', 'delta_stf_llm', 'delta_diff_0',
       'delta_diff_1', 'delta_diff_2', 'delta_diff_3', 'delta_diff_4',
       'delta_diff_5', 'delta_diff_6', 'delta_diff_7', 'delta_diff_8',
       'delta_diff_9'],
      dtype='object')

In [5]:
# calculate variablity before and after the survey and also using the simulations
var_df = master_df[['Question',"GROUP","BeforeNumber",'AfterNumber', 'run_0', 'run_1', 'run_2', 'run_3', 'run_4', 'run_5', 'run_6',
       'run_7', 'run_8', 'run_9']].groupby(['Question',"GROUP"]).var().reset_index()
var_df.head()

,Question,GROUP,BeforeNumber,AfterNumber,run_0,run_1,run_2,run_3,run_4,run_5,run_6,run_7,run_8,run_9
0,Q1,1,3.670330,2.615385,3.873626,4.840659,6.642857,2.994505,3.565934,3.873626,3.038462,2.862637,4.840659,5.324176
1,Q1,2,6.429167,3.029167,6.462500,9.362500,7.695833,8.062500,7.533333,7.329167,5.895833,4.129167,7.495833,9.129167
2,Q1,3,6.641026,4.641026,8.025641,7.692308,3.692308,6.769231,9.256410,7.589744,10.769231,4.256410,3.410256,3.692308
3,Q1,4,3.174242,3.454545,6.083333,4.810606,2.265152,2.265152,1.969697,1.333333,2.265152,2.606061,3.295455,2.000000
4,Q1,5,7.602564,7.307692,6.589744,9.076923,6.410256,7.692308,6.666667,6.230769,7.474359,5.641026,9.076923,7.692308


In [12]:
# cacculate mean variance across simulations
var_df['SimulatedVariance'] = var_df[['run_0', 'run_1', 'run_2', 'run_3', 'run_4', 'run_5', 'run_6',
       'run_7', 'run_8', 'run_9']].mean(axis=1)

In [13]:
q_var_df =  var_df[var_df["Question"].isin(bestQuestions)]

In [14]:
q_var_df.head()

,Question,GROUP,BeforeNumber,AfterNumber,run_0,run_1,run_2,run_3,run_4,run_5,run_6,run_7,run_8,run_9,SimulatedVariance
320,Q2H,1,11.897436,12.397436,3.935897,10.397436,9.397436,10.474359,12.833333,6.974359,9.192308,8.410256,10.474359,10.192308,9.228205
321,Q2H,2,9.795833,7.716667,3.095833,6.783333,7.450000,7.583333,8.229167,7.195833,7.583333,7.333333,7.066667,8.095833,7.041667
322,Q2H,3,8.379121,9.170330,7.142857,7.340659,4.769231,4.686813,2.901099,6.989011,4.071429,4.686813,4.219780,3.104396,4.991209
323,Q2H,4,12.787879,5.909091,3.242424,9.151515,11.901515,13.636364,10.515152,10.333333,9.356061,4.969697,3.242424,13.537879,8.988636
324,Q2H,5,13.181818,7.174242,3.000000,8.333333,7.719697,9.636364,5.000000,10.446970,11.454545,2.969697,10.568182,10.810606,7.993939


In [9]:
from scipy.stats import ks_2samp, wasserstein_distance, gaussian_kde
from scipy.spatial.distance import jensenshannon


results = []
for question in bestQuestions:
    q_df = q_var_df[q_var_df['Question'] == question]
    human_var  = q_df['AfterNumber']
    
    ks_stats, ks_pvals = [], []
    w_dists, js_divs = [], []

    for i in range(10):
        sim_var = q_df[f'run_{i}']

        if human_var.equals(sim_var):
            ks_stat, ks_pval, w_dist, js = 0.0, 1.0, 0.0, 0.0
    
        else: 
            ks_stat, ks_pval = ks_2samp(human_var, sim_var)
            w_dist = wasserstein_distance(human_var, sim_var)
            # 2. Continuous JS Calculation using KDE
            # Define a shared evaluation grid across the range of both datasets
            mins = min(human_var.min(), sim_var.min())
            maxs = max(human_var.max(), sim_var.max())
            grid = np.linspace(mins, maxs, 1000) 

            # Fit KDEs
            # Note: Use a small floor (1e-10) to avoid division by zero/log issues
            kde_h = gaussian_kde(human_var)(grid) + 1e-10
            kde_l = gaussian_kde(sim_var)(grid) + 1e-10

            # Normalize to ensure they are valid probability distributions (sum to 1)
            p_h = kde_h / kde_h.sum()
            p_l = kde_l / kde_l.sum()

            # Calculate Jensen-Shannon distance
            js = jensenshannon(p_h, p_l)
            
            ks_stats.append(ks_stat)
            ks_pvals.append(ks_pval)
            w_dists.append(w_dist)
            js_divs.append(js)
    
    print(f"Question: {question}")
    results.append({
        'question': question,
        'human_mean_var': human_var.mean(),
        'llm_mean_var': q_df['SimulatedVariance'].mean(),
        'ks_D_mean': np.mean(ks_stats),
        'ks_D_sd': np.std(ks_stats),
        'ks_p_mean': np.mean(ks_pvals),
        'wasserstein_mean': np.mean(w_dists),
        'wasserstein_sd': np.std(w_dists),
        'js_mean': np.mean(js_divs),
        'js_sd': np.std(js_divs),
    })
results_df = pd.DataFrame(results)

Question: Q5J
Question: Q5E
Question: Q3A
Question: Q3F
Question: Q4E
Question: Q3B
Question: Q2H
Question: Q5B
Question: Q3C
Question: Q5D


In [10]:
results_df.head(10)

,question,human_mean_var,llm_mean_var,ks_D_mean,ks_D_sd,ks_p_mean,wasserstein_mean,wasserstein_sd,js_mean,js_sd
0,Q5J,4.195265,3.753222,0.2375,0.049054,0.274790,0.821154,0.161338,0.127554,0.033317
1,Q5E,2.013225,1.529455,0.2225,0.045346,0.334025,0.611073,0.108713,0.191637,0.036711
2,Q3A,11.654131,10.604627,0.2575,0.110708,0.362507,2.125698,0.742021,0.181605,0.056494
3,Q3F,8.747424,8.619168,0.1475,0.043946,0.750431,0.854803,0.219604,0.104576,0.043273
4,Q4E,5.174852,5.049402,0.1875,0.046435,0.530612,0.837592,0.164960,0.109417,0.027598
5,Q3B,10.646953,11.218043,0.2400,0.058310,0.284318,1.646844,0.368773,0.156502,0.037260
6,Q2H,7.717962,9.264305,0.2400,0.066332,0.298454,1.762691,0.816616,0.139685,0.054064
7,Q5B,9.994794,9.365717,0.1800,0.041533,0.543819,0.839973,0.271702,0.113406,0.046723
8,Q3C,8.416499,7.672472,0.2275,0.097756,0.442304,1.403017,0.793692,0.132782,0.082945
9,Q5D,6.479656,7.898961,0.2200,0.073993,0.393609,1.466581,0.585941,0.133389,0.055180


In [11]:
results_df

,question,human_mean_var,llm_mean_var,ks_D_mean,ks_D_sd,ks_p_mean,wasserstein_mean,wasserstein_sd,js_mean,js_sd
0,Q5J,4.195265,3.753222,0.2375,0.049054,0.274790,0.821154,0.161338,0.127554,0.033317
1,Q5E,2.013225,1.529455,0.2225,0.045346,0.334025,0.611073,0.108713,0.191637,0.036711
2,Q3A,11.654131,10.604627,0.2575,0.110708,0.362507,2.125698,0.742021,0.181605,0.056494
3,Q3F,8.747424,8.619168,0.1475,0.043946,0.750431,0.854803,0.219604,0.104576,0.043273
4,Q4E,5.174852,5.049402,0.1875,0.046435,0.530612,0.837592,0.164960,0.109417,0.027598
5,Q3B,10.646953,11.218043,0.2400,0.058310,0.284318,1.646844,0.368773,0.156502,0.037260
6,Q2H,7.717962,9.264305,0.2400,0.066332,0.298454,1.762691,0.816616,0.139685,0.054064
7,Q5B,9.994794,9.365717,0.1800,0.041533,0.543819,0.839973,0.271702,0.113406,0.046723
8,Q3C,8.416499,7.672472,0.2275,0.097756,0.442304,1.403017,0.793692,0.132782,0.082945
9,Q5D,6.479656,7.898961,0.2200,0.073993,0.393609,1.466581,0.585941,0.133389,0.055180
